# CS 540 – Movie Recommender from Scratch

**Marcus Birkenkrahe, Catholic Polytechnic University**

This notebook builds a simple movie recommender system using only NumPy.
There is no scikit-learn here — the goal is to see *exactly* what is
happening mathematically, before the library hides it from you.

This exercise was inspired by the DataCamp course
[*Understanding Machine Learning*, Chapter 1](https://campus.datacamp.com/courses/understanding-machine-learning/what-is-machine-learning?ex=2).

## The idea

Each movie is described by a small set of binary features:
`[action, comedy, dark_tone, fast_paced]`.
A value of `1` means the movie has that quality; `0` means it does not.

When you make a query — "I want a dark action film" — you provide the
same kind of vector: `[1, 0, 1, 0]`.

The recommender finds the movies whose feature vectors point in the same
*direction* as your query. That direction is measured by **cosine similarity**:

> Two vectors are similar if the angle between them is small.
> Cosine similarity = 1 means perfect match; 0 means no overlap at all.

$$\cos\theta = \frac{\mathbf{A} \cdot \mathbf{B}}{|\mathbf{A}|\ |\mathbf{B}|}$$

Where:
- $\mathbf{A} \cdot \mathbf{B}$ is the **dot product**: multiply corresponding elements and sum
- $|\mathbf{A}|$ is the **magnitude** of A: $\sqrt{\sum A_i^2}$
- $|\mathbf{B}|$ is the **magnitude** of B: $\sqrt{\sum B_i^2}$

**Worked example:** query `[1, 0, 1, 0]` vs. *The Dark Knight* `[1, 0, 1, 1]`

$$\cos\theta = \frac{(1)(1)+(0)(0)+(1)(1)+(0)(1)}{\sqrt{1^2+0^2+1^2+0^2}\ \times\ \sqrt{1^2+0^2+1^2+1^2}} = \frac{2}{\sqrt{2}\ \times\ \sqrt{3}} = \frac{2}{\sqrt{6}} \approx 0.816 = 81.6\%$$

*Illustration: cosine similarity in 2D, using only `action` and `dark_tone` as features.*

![Cosine similarity 2D](https://raw.githubusercontent.com/birkenkrahe/cs540/main/img/cosine_similarity_2d.svg)

In [ ]:
import numpy as np

## Movie database

Each film is represented as a 4-dimensional binary vector:
`[action, comedy, dark_tone, fast_paced]`

In [ ]:
movies = [
    {"title": "The Dark Knight (2008)",    "features": [1, 0, 1, 1]},
    {"title": "Mad Max: Fury Road (2015)", "features": [1, 0, 1, 1]},
    {"title": "Baby Driver (2017)",        "features": [1, 0, 0, 1]},
    {"title": "John Wick (2014)",          "features": [1, 0, 1, 1]},
    {"title": "Speed Racer (2008)",        "features": [1, 0, 0, 1]},
    {"title": "Hot Fuzz (2007)",           "features": [1, 1, 1, 1]},
    {"title": "In Bruges (2008)",          "features": [0, 1, 1, 0]},
    {"title": "The Nice Guys (2016)",      "features": [1, 1, 1, 0]},
    {"title": "Four Weddings (1994)",      "features": [0, 1, 0, 0]},
    {"title": "Superbad (2007)",           "features": [0, 1, 0, 0]},
    {"title": "This Is the End (2013)",    "features": [0, 1, 1, 0]},
    {"title": "Knives Out (2019)",         "features": [0, 1, 1, 0]},
]

titles = [m["title"] for m in movies]
X = np.array([m["features"] for m in movies])  # shape: (12, 4)
print(f"{len(movies)} movies, feature matrix shape: {X.shape}")

## Cosine similarity and recommender functions

In [ ]:
def cosine_similarity(query, matrix):
    dot    = matrix @ query
    mag_q  = np.linalg.norm(query)
    mag_m  = np.linalg.norm(matrix, axis=1)
    denom  = mag_m * mag_q
    return np.where(denom == 0, 0.0, dot / denom)

def recommend(action=0, comedy=0, dark_tone=0, fast_paced=0, top_n=5):
    query  = np.array([action, comedy, dark_tone, fast_paced], dtype=float)
    scores = cosine_similarity(query, X)
    ranked = np.argsort(scores)[::-1][:top_n]
    print(f"\nQuery: action={action}, comedy={comedy}, "
          f"dark_tone={dark_tone}, fast_paced={fast_paced}\n")
    print(f"{'Rank':<6} {'Title':<35} {'Match':>6}")
    print("-" * 50)
    for rank, idx in enumerate(ranked, 1):
        print(f"{rank:<6} {titles[idx]:<35} {scores[idx]*100:>5.1f}%")

## Try it — three example queries

In [ ]:
recommend(action=1, dark_tone=1)                          # dark action film
recommend(action=1, comedy=1, dark_tone=1, fast_paced=1)  # fast-paced dark action-comedy
recommend(comedy=1, dark_tone=1)                          # dark comedy

---
## Your turn

Work through the tasks below. For each one, add a new code cell
(or markdown cell where indicated) directly beneath the task.

**Task 1 — Run and observe**

Run the three example queries above. Pick one result that surprised
you and one that made perfect sense. Write a markdown cell explaining
why each result did or did not match your intuition.

**Task 2 — Change the query**

Write two new `recommend()` calls with different genre combinations
of your own choice. For each query, identify the top result and
explain in one sentence why it scored highest — referring specifically
to the feature vector values.

**Task 3 — Add your own movies**

Add at least 3 movies of your own to the `movies` list above.
Assign binary feature values `[action, comedy, dark_tone, fast_paced]`
honestly — think about each film before you assign. Re-run the
database setup cell and then run a query where you expect at least
one of your new movies to appear in the top 5.

**Task 4 — Reflect**

In a markdown cell, answer these two questions in 3–5 sentences total:

1. Only four binary features describe every movie. What important
   qualities are missing — and what would happen to the recommendations
   if you added them?
2. There are no IF-THEN rules anywhere in this code. What is actually
   doing the deciding — and where exactly in the code does that happen?

**Submission**

Share this notebook (anyone with the link can view) and submit the
URL to the Teams assignment.